In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
import seaborn as sns

In [ ]:
%pwd

In [ ]:
df = pd.read_csv("../../data/aita_top_subs.csv")   # must contain df['selftext']
df = df.dropna(subset=['selftext'])
df = df.reset_index(drop=True)

# Classification Validation

When you train a classifier, you need to check two things:

### 1. How well it performs

Because we have labels, we can use standard metrics:
- Accuracy – overall percentage correct
- Precision – of the items predicted positive, how many were actually positive?
- Recall – of the real positives, how many did the model find?
- F1 – balances precision + recall
- Confusion Matrix – shows exactly where the model is making mistakes
- Baseline – always compare to a simple majority-class model

If your model doesn’t beat the baseline, it’s not learning anything useful.

### 2. How stable the performance is (sensitivity)

Good scores once don’t mean much unless they’re consistent.

Check whether your metrics stay similar when you:
- change the random seed
- use different train/test splits
- train on different subsamples of the data

If the numbers swing wildly, the model is unstable and your conclusions may not hold.

- Performance tells you how good the model is.
- Sensitivity checks tell you whether you can trust the results.

In [ ]:
df = df[df['flair_css_class'].isin(['ass', 'not'])]

In [ ]:
df.shape

In [ ]:
# Train/test split

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df_us['selftext']
y = df_us['flair_css_class']  # assume they have some label

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=200))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

In [ ]:
# Train/test split

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df['selftext']
y = df['flair_css_class']  # assume they have some label

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=200))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

In [ ]:
# Undersampling balances the data but loses information. The model becomes less stable and less predictive.
print(classification_report(y_test, pred))

In [ ]:
# compare to baseline
print("Majority class baseline:", y_test.value_counts().max() / len(y_test))

"If I always say ‘not,’ I’ll be right most of the time"

### Solutions
Undersampling the majority

In [ ]:
df_us = df.groupby('flair_css_class').sample(n=231, random_state=42)
df_us.shape

In [ ]:
# Train/test split

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df_us['selftext']
y = df_us['flair_css_class']  # assume they have some label

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=200))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

In [ ]:
# Undersampling balances the data but loses information. The model becomes less stable and less predictive.
print(classification_report(y_test, pred))

Oversampling the minority:

In [ ]:
from sklearn.utils import resample

df_major = df[df.flair_css_class == "not"]
df_minor = df[df.flair_css_class == "ass"]
df_minor_up = resample(df_minor, replace=True, n_samples=len(df_major))

df_os = pd.concat([df_major, df_minor_up])

In [ ]:
# Train/test split

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df_os['selftext']
y = df_os['flair_css_class']  # assume they have some label

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=200))
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

In [ ]:
# Oversampling keeps all data and balances the classes, leading to better performance and better recall on the minority class.
print(classification_report(y_test, pred))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix (TN, FP, FN, TP)")

- If your model does worse than majority class, it’s meaningless.
- If it’s only a few points better, consider whether features are capturing anything.
- Look at per-class F1, not accuracy.
- Check overfitting by seeing if train accuracy ≫ test accuracy.


# Validation for Embeddings & Bias (WEAT-style sanity checks)

In [ ]:
df = pd.read_csv("../../data/aita_top_comments.csv")   # must contain df['selftext']

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

emb = model.encode(df['body'].tolist(), show_progress_bar=True)

In [ ]:
def compute_bias_score(model, target_A, target_B, attrs_pos, attrs_neg):
    # A.shape → (10, 384) if MiniLM has 384 dimensions -> one embedding vector per word
    A = model.encode(target_A)
    B = model.encode(target_B)
    pos = model.encode(attrs_pos)
    neg = model.encode(attrs_neg)

    def mean_cosine(t, a):
        return cosine_similarity(t, a).mean()

    # Are A-words closer to positive words than to negative words? If stat_A > 0 → A is more positive
    stat_A = mean_cosine(A, pos) - mean_cosine(A, neg)

    # Are B-words closer to positive or negative concepts? If stat_B > 0 → B is more positive
    stat_B = mean_cosine(B, pos) - mean_cosine(B, neg)
    return stat_A - stat_B

Sensitivity checks are tests that show how fragile or stable your results are when you make small, reasonable changes to:
- the data
- the sampling
- the random seed
- the method
- the input word lists
- the model parameters

They answer the question:

“If I rerun this with slightly different inputs, do I get the same result?”

If yes → your finding is robust.
If no → your result is sensitive, meaning it may not be trustworthy.


✅ What bootstrapping is doing here

Bootstrapping is a sensitivity check:
we want to see whether the bias score is stable or if it changes a lot when we slightly change the input.

In your setup, the “input” being perturbed is:
	•	the attribute word lists (for embedding bias)
	•	the documents belonging to a topic (for BERTopic topic stability)

So:

Bootstrapping = repeatedly recomputing the bias/topic score on many random subsets of the data to see how much it changes.

If the score stays the same → stable, trustworthy result
If the score jumps around → unstable, unreliable result

⸻

✅ Bootstrapping in your embedding bias test

You compute a bias score like: `bias = (A closer to pos – A closer to neg) – (B closer to pos – B closer to neg)`

Then you do:
	•	randomly sample 80% of your pos/neg attribute words
	•	recompute the bias score
	•	repeat N = 30 or 50 times
	•	collect all the scores
	•	plot them

This tells you:

✔ Is the bias real?

Strong bias → the distribution shifts away from zero.

✔ Is the bias robust?

Stable bias → all bootstrap samples give similar scores.

✔ Or is it noise?

Unstable bias → scores jump from negative to positive depending on random sampling.

So bootstrapping answers the question:

If we didn’t pick exactly these attribute words, would the bias test give the same conclusion?

This is exactly the main critique of WEAT-style bias tests — they depend on small, hand-written word lists — so bootstrapping is the fix.


In [ ]:
# Target groups -- which gender-coded behavior terms are more associated with positive vs. negative evaluations
target_A = [
    "aggressive", "dominant", "controlling", "forceful",
    "reckless", "selfish", "angry", "hotheaded",
    "intimidating", "demanding"
]

target_B = [
    "dramatic", "emotional", "manipulative", "passive-aggressive",
    "clingy", "overreacting", "moody", "nagging",
    "jealous", "sensitive"
]

In [ ]:
attrs_pos = [
    "good", "kind", "fair", "reasonable", "balanced",
    "thoughtful", "calm", "patient", "supportive", "honest",
    "respectful", "generous", "mature", "trustworthy"
]

attrs_neg = [
    "bad", "wrong", "cruel", "toxic", "unfair", "mean", "abusive",
    "hurtful", "disrespectful", "manipulative", "selfish",
    "aggressive", "hostile", "rude", "irresponsible", "immature",
    "dangerous", "violent", "exploitative", "controlling"
]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# This tests whether the bias estimate is stable or noise.

import random

def bootstrap_bias(model, target_A, target_B, attrs_pos, attrs_neg, n_boot=50, sample_size=0.8):
    scores = []
    for i in range(n_boot):
        pos_sample = random.sample(attrs_pos, int(len(attrs_pos) * sample_size))
        neg_sample = random.sample(attrs_neg, int(len(attrs_neg) * sample_size))
        score = compute_bias_score(model, target_A, target_B, pos_sample, neg_sample)
        scores.append(score)
    return scores

scores = bootstrap_bias(model, target_A, target_B, attrs_pos, attrs_neg)

plt.hist(scores)
plt.title("Bootstrap Distribution of Bias Score")
plt.show()

print("Mean:", np.mean(scores))
print("Std:", np.std(scores))

In [ ]:
# Target groups
target_A = [
    "respectful", "considerate", "responsible",
    "thoughtful", "reasonable", "mature",
    "kind", "communicative"
]

target_B = [
    "selfish", "entitled", "rude",
    "manipulative", "inconsiderate",
    "toxic", "lazy", "immature"
]

In [ ]:
attrs_pos = [
    "good", "fair", "kind", "just", "right",
    "reasonable", "supportive", "considerate"
]

attrs_neg = [
    "bad", "wrong", "unfair", "mean", "cruel",
    "selfish", "toxic", "disrespectful"
]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# This tests whether the bias estimate is stable or noise.

import random

def bootstrap_bias(model, target_A, target_B, attrs_pos, attrs_neg, n_boot=50, sample_size=0.8):
    scores = []
    for i in range(n_boot):
        pos_sample = random.sample(attrs_pos, int(len(attrs_pos) * sample_size))
        neg_sample = random.sample(attrs_neg, int(len(attrs_neg) * sample_size))
        score = compute_bias_score(model, target_A, target_B, pos_sample, neg_sample)
        scores.append(score)
    return scores

scores = bootstrap_bias(model, target_A, target_B, attrs_pos, attrs_neg)

plt.hist(scores)
plt.title("Bootstrap Distribution of Bias Score")
plt.show()

print("Mean:", np.mean(scores))
print("Std:", np.std(scores))

Our data is ... 

1. Centered around 0 -- No differential association.
2. Narrow spread (low STD) -- The result is stable and does not depend on which 80% sample of attributes you pick.
3. No multi-modal structure -- No hidden subgroups or weird split patterns.
4. Resampling never pushes the score far from zero -- No evidence of latent bias that appears only with certain word subsets.

That means the model shows no meaningful association difference between the two groups for these attribute sets.

In [ ]:
target_A = ["husband", "boyfriend", "fiancé", "partner", "guy", "man"]
target_B = ["wife", "girlfriend", "fiancée", "partner", "girl", "woman"]

attrs_pos = [
    "cheated", "cheating", "slept", "affair", "flirted",
    "texting", "messaging", "lying", "gaslighting",
    "unfaithful", "dishonest", "betrayal"
]

attrs_neg = [
    "loyal", "faithful", "honest", "trustworthy",
    "supportive", "committed", "respectful", "caring"
]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# This tests whether the bias estimate is stable or noise.

import random

def bootstrap_bias(model, target_A, target_B, attrs_pos, attrs_neg, n_boot=50, sample_size=0.8):
    scores = []
    for i in range(n_boot):
        pos_sample = random.sample(attrs_pos, int(len(attrs_pos) * sample_size))
        neg_sample = random.sample(attrs_neg, int(len(attrs_neg) * sample_size))
        score = compute_bias_score(model, target_A, target_B, pos_sample, neg_sample)
        scores.append(score)
    return scores

scores = bootstrap_bias(model, target_A, target_B, attrs_pos, attrs_pos)

plt.hist(scores)
plt.title("Bootstrap Distribution of Bias Score")
plt.show()

print("Mean:", np.mean(scores))
print("Std:", np.std(scores))

# Validating Topic Models: Sensitivity and Stability

### What makes a topic model “good”?

Good topic models should be **meaningful**, **interpretable**, and **reproducible**.  

A topic model is “good” only if it supports the business or research question.

Three criteria matter most:

1. Topics must be interpretable by a human: 

You (or a stakeholder) can look at the top words and representative docs and say:

- “Ah, this is the topic about pet problems.”
- “This is the topic about financial conflicts.”
- “This is the topic about weddings.”

If a topic is too broad, too mixed, or too small, it’s “bad.”


2. Topics must be stable enough to trust

Not perfectly stable, but consistent enough to make decisions. Analysts usually check:
- topic size
- topic coherence
- cluster compactness
- recurrence across subsamples
- does it split into garbage topics or meaningful ones?

3. Topics must be actionable

- if the company/product/policy team can do something with the results
- if the model helps the research question
- if it can be operationalized

4. What do analysts do when a topic is BAD?

They do not hand-delete it! They do one of these:

- Merge small/noisy topics into larger ones (common) with BERTopic's `reduce_topics` — this is the industry standard.
- Collapse all noisy topics into an “Other” bucket
- Adjust model parameters and re-run -- increase `min_cluster_size`, adjust vectorizer, switch embeddings, adjust preprocessing
- Ignore bad topics downstream --select the 10–20 most stable topics as features and ignore the tail 
- Use topic modeling as a first pass, then manually label meta-themes based on clusters. Topic models often serve as codebook generators, not final classifiers.

Topic models need to be VALIDATED and REFINED! 

In [ ]:
# Grab data
df = df[~df["selftext"].str.strip().isin(["[deleted]", "[removed]", ""])]

In [ ]:
# Minimal text cleaning (retain original linguistic structure for BERTopic)
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    return text.strip()
    
df["clean_text"] = df["selftext"].apply(clean_text)

df.reset_index(drop=True, inplace=True)

In [ ]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

docs = df["clean_text"].tolist()

# vectorizer: clean topic words (does not affect embeddings)
extra_stop = {"she","her","he","him","they","them","it","its","that","this","these","those"}
vectorizer_model = CountVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | extra_stop),
    ngram_range=(1,2),
    min_df=5
)

umap_model = UMAP(
    n_neighbors=50,      # ↑ neighborhood = fewer outliers
    n_components=15,     # ↑ dims = less over-compression
    metric="cosine",
    random_state=0
)

hdb_model = HDBSCAN(
    min_cluster_size=40, # ↑ size = fewer tiny shards
    min_samples=1,       # ↓ strictness = fewer "-1"
    metric="euclidean",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",        # stronger than all-MiniLM-L6-v2
    umap_model=umap_model,
    hdbscan_model=hdb_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    nr_topics=None,                              # don’t merge yet
)

topics, probs = topic_model.fit_transform(docs)

In [ ]:
topic_model.get_topic_info()

In [ ]:
# assign topics back to dataframe
df["topic"] = topics

Now we can do topic mapping, based on domain knowledge and reviewing the top words of each topic:

In [ ]:
macro_map = {
    # Marriage / Domestic Partnership
    0: "Marriage / Domestic",
    1: "Marriage / Domestic",
    23: "Marriage / Domestic",
    25: "Marriage / Domestic",
    32: "Marriage / Domestic",
    35: "Marriage / Domestic",
    37: "Marriage / Domestic",

    # Parenting / Kids / Childcare
    4: "Parenting / Kids",
    8: "Parenting / Kids",
    12: "Parenting / Kids",
    22: "Parenting / Kids",
    26: "Parenting / Kids",
    33: "Parenting / Kids",

    # Weddings / Engagements
    2: "Weddings / Engagements",
    28: "Weddings / Engagements",

    # Workplace / Career
    13: "Workplace / Career",
    30: "Workplace / Career",

    # Housing / Roommates / Neighbors
    10: "Housing / Roommates",
    19: "Housing / Roommates",
    29: "Housing / Roommates",

    # Friends / Social / Dating
    18: "Friends / Social",
    24: "Friends / Social",
    17: "Friends / Social",

    # Money / Finances / Inheritance
    5: "Money / Finances",
    13: "Money / Finances",
    20: "Money / Finances",

    # Food / Diet / Cooking
    3: "Food / Diet",
    34: "Food / Diet",

    # Pets / Animals
    14: "Pets / Animals",
    31: "Pets / Animals",

    # Holidays / Gifts / Events
    21: "Holidays / Gifts",
    38: "Holidays / Gifts",

    # Health / Medical / Bodies
    6: "Health / Medical",
    15: "Health / Medical",
    39: "Health / Medical",
    40: "Health / Medical",

    # Religion / Identity / LGBTQ
    9:  "Religion / Identity",
    21: "Religion / Identity",   # if you want LGBTQ+ under identity

    # Alcohol / Drinking
    42: "Alcohol / Drinking",

    # Cars / Driving
    43: "Cars / Driving",

    # Meta / Reddit / AITA itself
    44: "Meta / AITA",
}

df["macro_topic"] = df["topic"].map(macro_map).fillna("Other")

In [ ]:
df["macro_topic"].value_counts()

## Validation approaches 

We now can use three complementary validation checks that move from *local* (inside a topic) to *global* (across many model runs):

- **Cluster coherence** → Is the topic *real*? (embedding tightness)  
- **Topic coherence** → Does the topic *make sense*? (word-level meaning)  
- **Topic stability** → Does the topic *hold up*? (reproducibility across runs)

Together these checks give a complete view of topic quality.

## 1. Cluster Coherence (Embedding-Level Tightness)

**Question:** *Do the documents inside topics actually form a semantic cluster?*

This is an **embedding-level** check: we look at how close the topic’s documents are to one another in vector space.  

**How we test this:**

1. Extract all documents assigned to a topic.  
2. Randomly sample 80% of those documents (bootstrap).  
3. Compute embeddings for the sampled documents (same as when running BERTopic -- one embedding per post).  
4. Compute the centroid of those embeddings.  
5. Measure the **mean cosine similarity** of the documents to that centroid.  
6. Repeat ~30 times to get a **distribution** of coherence scores.

If the distribution is:
- **high and tight** → the topic is semantically well-formed.  
- **low or very wide** → the topic is unstable or not a real cluster.


### What is Bootstrapping? 

Bootstrapping is a simple resampling technique used to estimate how stable or reliable a statistic is. Instead of relying on a single sample of data, you repeatedly draw new samples with replacement from the original dataset, compute the statistic each time, and observe how much it varies across those resampled runs. 

If the statistic stays similar across many bootstrap samples, it’s considered stable; if it fluctuates a lot, it’s unstable.

In the context of topic modeling, bootstrapping lets us see whether a topic remains semantically tight when we repeatedly resample the documents within it.

In [ ]:
import random
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def cluster_coherence_bootstrap(model, topic_id, df, n_boot=20, sample_frac=0.8):
    """
    Cluster Coherence Under Resampling.

    Measures how tightly a topic's document embeddings stay clustered
    when we repeatedly resample its documents.
    """

    # topic assignments from BERTopic
    topics = model.topics_

    # indices of docs that belong to this topic
    idx = [i for i, t in enumerate(topics) if t == topic_id]

    # extract those documents
    docs = df["clean_text"].iloc[idx].astype(str).tolist()

    if len(docs) < 20:
        print("Topic too small for bootstrapping.")
        return None

    scores = []
    k = int(sample_frac * len(docs))   # how many docs we sample each bootstrap run

    for _ in range(n_boot):
        sample_docs = random.sample(docs, k)
        emb = model.embedding_model.embed_documents(sample_docs)

        # centroid of sampled embeddings
        centroid = emb.mean(axis=0)

        # mean cosine similarity to centroid = cluster tightness
        coherence = cosine_similarity(emb, centroid.reshape(1, -1)).mean()
        scores.append(coherence)

    return scores

In [ ]:
# This will take a while to run! 

topic_ids = sorted(set(topic_model.topics_) - {-1})   # drop outliers if needed

topic_scores = {}

for tid in topic_ids:
    scores = cluster_coherence_bootstrap(topic_model, topic_id=tid, df=df)
    if scores is not None:
        topic_scores[tid] = scores

In [ ]:
import pandas as pd
summary = []

for tid, scores in topic_scores.items():
    summary.append({
        "topic_id": tid,
        "mean": np.mean(scores),
        "std": np.std(scores),
        "cv": np.std(scores) / np.mean(scores),
        "n_docs": sum(np.array(topic_model.topics_) == tid),
    })

summary_df = pd.DataFrame(summary)

In [ ]:
plt.hist(summary_df["mean"], bins=20, edgecolor="black")
plt.title("Distribution of Topic Coherence (Bootstrap Mean)")
plt.xlabel("Mean Coherence")
plt.ylabel("Count of Topics")
plt.show()

In [ ]:
# Look at topics with lowest mean coherence (weakest topics)
summary_df.sort_values("mean").head(10)


What these statistics mean

For each topic, we bootstrap its documents and measure how tightly they cluster in embedding space.
- `mean` = average coherence (higher = more stable topic)
- `std` = variation across bootstrap samples
- `cv` = std / mean (relative instability; lower = better)
- `n_docs` = number of documents assigned to the topic

These metrics tell us whether a topic is coherent, stable, and meaningful, or whether it’s fragile or over-clustered.


In [ ]:
topic_model.get_topic(29)

In [ ]:
fig = topic_model.visualize_documents(df["clean_text"], topics=[29])
fig.show()

In [ ]:
topic_model.get_representative_docs(29)

This actually still looks interpretable.

Based on the quality of these documents, you should decide if you want to keep them, remove them from analysis, or re-run the model with adjusted parameters to improve the 

## 2. Topic Coherence

**Question:** *Do the top words in a topic actually make sense together?*

Topics that humans think are good tend to have top words that co-occur frequently in the same documents.

Thus coherence = a numeric proxy for top words belonging together in actual text.

- Uses word-based coherence (C_V, NPMI, etc.).
- Single model, no resampling.
- Evaluates human interpretability of the topic.

This is the standard “coherence” metric in topic modeling papers.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
import numpy as np

# ---- Extract clean topics ----
topic_ids = [t for t in topic_model.get_topics().keys() if t != -1]
topics_words = []
for t in topic_ids:
    words = [w for w, _ in topic_model.get_topic(t)]
    if len(words) >= 3:               # filter topics too small to evaluate
        topics_words.append(words[:10])

# ---- Tokenize docs ----
texts = [simple_preprocess(str(x)) for x in df["clean_text"].astype(str)]

dictionary = Dictionary(texts)

In [ ]:
# ---- Compute coherence ----
cm = CoherenceModel(
    topics=topics_words,
    texts=texts,
    dictionary=dictionary,
    coherence="c_v",          
)

topic_scores = cm.get_coherence_per_topic()
overall = np.nanmean(topic_scores)


We are using `c_v` (coherence_v):
It asks: “How often do these top words appear together in sliding windows?”

- Looks at co-occurrence within windows across the corpus
- Uses cosine similarity between word vectors derived from co-occurrence
- Highly sensitive to word frequency, window size, and rare words
- Often inflates scores with BERTopic’s C-TF-IDF top words (the reason you saw many 1’s)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(topic_scores, bins=10, edgecolor='black')
plt.title("Topic Coherence Distribution (c_v)")
plt.xlabel("Coherence Score")
plt.ylabel("Number of Topics")
plt.show()

### We have a plot, so what?

A "good" plot does NOT mean our model is better -- just that the metric is not lying to us.

A good plot helps us find the bad topics — the ones we should inspect manually. We should look at top words and see if these topics are interpretable.

In [ ]:
# Pair each topic with its coherence score
topic_scores_map = list(zip(valid_topic_ids, topic_scores))

# Sort from worst → best
topic_scores_map_sorted = sorted(topic_scores_map, key=lambda x: x[1])

In [ ]:
print("Worst 10 topics by coherence:")
for tid, score in topic_scores_map_sorted[:10]:
    print(f"Topic {tid} | score={score:.4f}")
    print("Top words:", [w for w, _ in topic_model.get_topic(tid)])
    print()

For each low-coherence topic, we can now merge it with other topics.

We should **NOT** rely on BERTopic’s automatic merging for this. Automatic merging (reduce_topics) merges based on c-TF-IDF similarity, which is purely lexical. But our topics are conceptually similar but lexically different.

In [ ]:
weak_topic_macro_map = {
    34: "Marriage / Domestic",
    31: "Parenting / Kids",
    22: "Parenting / Kids",     # or "Health / Medical"
    25: "Parenting / Kids",
    35: "Parenting / Kids",     # or create "Family / Siblings"
    32: "Parenting / Kids",
    23: "Friends / Social",
    28: "Housing / Roommates",
    17: "Friends / Social",
    21: "Holidays / Gifts",
}
df["macro_topic"] = df["topic"].map(weak_topic_macro_map).fillna(df["macro_topic"])

In [ ]:
df["macro_topic"].value_counts()

## 3. Topic Stability (Resampling Robustness)

**Question:**  
*If we train BERTopic on slightly different samples of the data, do the same topics re-appear?*

This tests whether each topic is a real, reproducible structure in the corpus or a sampling artifact.

1. Randomly sample 80% of the dataset several times.
2. Train a new BERTopic model on each subset.
3. For each topic in the full model:
    - Take its top-10 words
	- Find the topic in each subset model whose top-10 words overlap the most
	- Record that overlap score
4. The mean overlap across resampled models is the topic’s stability score.

Interpretation
- High stability: The topic reliably emerges even when the data changes. These topics represent strong, persistent clusters.
- Low stability: The topic moves around, splits, or disappears. These are often incoherent topics, small clusters, or noise.


In [ ]:
# Extract all topics (topic_id -> list of (word, score))
base_topics_dict = topic_model.get_topics()

In [ ]:
def fit_on_subset(df, frac=0.8, seed=42):
    """
    Fit BERTopic on an 80% random subset of the data.
    Used to test global topic stability.
    """
    df_s = df.sample(frac=frac, random_state=seed)

    model_s = BERTopic()
    topics_s, _ = model_s.fit_transform(df_s["clean_text"])

    return model_s

In [ ]:
subset_models = []
seeds = [0, 13, 42, 77, 99]

for seed in seeds:
    m = fit_on_subset(df, frac=0.8, seed=seed)
    subset_models.append((seed, m))

In [ ]:
def best_topic_overlap(base_words, subset_model, n=10):
    """
    For one topic from the full model, find the subset topic
    whose top words overlap the most with it.
    """
    max_overlap = 0

    for t in subset_model.get_topics():
        subset_words = [w for w, _ in subset_model.get_topic(t)[:n]]
        overlap = len(set(base_words).intersection(subset_words))
        max_overlap = max(max_overlap, overlap)

    return max_overlap

In [ ]:
from collections import defaultdict
import numpy as np

topic_scores = defaultdict(list)

# Loop through subset models
for seed, m in subset_models:
    for topic_id, words in base_topics_dict.items():
        base_top10 = [w for w, _ in words[:10]]   # extract base top-10
        overlap = best_topic_overlap(base_top10, m)
        topic_scores[topic_id].append(overlap)

# Mean stability per topic
topic_stability = {
    t: np.mean(scores) for t, scores in topic_scores.items()
}

# Sort by stability (highest first)
sorted(topic_stability.items(), key=lambda x: -x[1])[:10]

In [ ]:
import matplotlib.pyplot as plt

all_scores = [np.mean(v) for v in topic_scores.values()]

plt.hist(all_scores, bins=10, edgecolor="black")
plt.title("Distribution of Topic Stability (Best-Match Word Overlap)")
plt.xlabel("Mean Word Overlap")
plt.ylabel("Number of Topics")
plt.show()

### Interpreting the Topic Stability Histogram

This histogram shows how stable the topics are **overall**, using the mean
best-match word overlap across the subset models.

Each bar corresponds to a range of stability scores (0–10 words shared).

**How to read the plot:**

- **Bars toward the right (6–10 overlap)**  
  → Many topics are consistently recovered across different subsets  
  → These topics are **robust and reliable**

- **Bars in the middle (3–5 overlap)**  
  → Moderately stable topics  
  → They appear in resampled models but with some variation in vocabulary

- **Bars toward the left (0–2 overlap)**  
  → Unstable topics  
  → These topics either disappear, split, or change semantics across runs  
  → Likely too small, noisy, or under-represented in the data

**What a “good” model looks like:**  
- Histogram skewed to the **right** (many high-scoring topics)  
- Few topics with stability near 0  

**What a “bad” model looks like:**  
- Histogram skewed to the **left**  
- Many unstable topics  
- Suggesting bad embeddings, too many topics, or insufficient data

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_stab = pd.DataFrame.from_dict(topic_scores, orient='index')
df_stab.columns = [f"seed_{s}" for s, _ in subset_models]

plt.figure(figsize=(10, 20))
sns.heatmap(df_stab, annot=False, cmap="viridis")
plt.title("Topic Stability Across Resampling")
plt.xlabel("Subset Run")
plt.ylabel("Topic ID")
plt.show()

### Interpreting the Topic Stability Heatmap

This heatmap shows how reproducible each topic is across 5 resampled BERTopic models.

Each row is a topic from the full model; each column is one 80% bootstrap subset.

The color represents the number of overlapping top-10 words (0–10) between the full-model topic and its best-matching topic in that subset model.

How to read the plot:
- Bright rows (high overlap across all columns): Topics whose top words reappear consistently. These are high-stability topics.
- Mixed rows (bright + dark cells): Topics that shift or split depending on the sample. These are moderately stable and should be interpreted cautiously.
- Dark rows (low overlap everywhere): Topics that rarely reappear. These are low-stability or spurious topics, often caused by small clusters, noisy vocabulary, or weak semantic structure.
- Highly irregular rows: Suggest that the topic is unstable due to low document count or because the model created overlapping or overly fine-grained clusters.

What to do with unstable topics:
- Treat them as weak or unreliable topics in reporting.
- Optionally merge or remove them.
- Or refit BERTopic with different parameters (e.g., larger min_cluster_size, fewer topics, or stronger preprocessing).

In [ ]:
import pandas as pd
import numpy as np

stability_df = pd.DataFrame.from_dict(topic_scores, orient="index")
stability_df.columns = [f"seed_{s}" for s in seeds]
stability_df["mean_stability"] = stability_df.mean(axis=1)
stability_df["min_stability"] = stability_df.min(axis=1)
stability_df["max_stability"] = stability_df.max(axis=1)

stability_df.sort_values("mean_stability")

Let's qualify the topics that have a low stability score (lower than 2) as "Unstable".

In [ ]:
unstable_topics = stability_df[stability_df["mean_stability"] < 2].index.tolist()

In [ ]:
df["topic_clean"] = df["topic"].apply(
    lambda t: "Unstable" if t in unstable_topics else t
)

In [ ]:
df["macro_topic_clean"] = df.apply(
    lambda row: "Unstable" if row["topic_clean"] == "Unstable" else row["macro_topic"],
    axis=1
)

In [ ]:
df["macro_topic_clean"].value_counts()

In [ ]:
df_plot = df[df["macro_topic_clean"] != "Other"]

df_plot["macro_topic_clean"].value_counts().sort_values().plot(
    kind="barh",
    figsize=(8,6)
)

plt.title("Number of Posts by Macro-Topic (Excluding Other)")
plt.xlabel("Count")
plt.ylabel("macro_topic_clean")
plt.show()

## What To Do With Topic Models 

Topic modeling is rarely the final product. It’s a foundation for decisions, reporting, and automated monitoring.

But topic models are not reliable by default: they can produce noisy, unstable, or meaningless clusters if you don’t validate them.
If you ship those results to leadership or stakeholders, you risk:
- drawing wrong conclusions
- misdiagnosing a problem
- making decisions based on artifacts rather than patterns

Once topics are stable and meaningful, you can safely do things like sentiment analysis.


In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
df["sentiment"] = df["selftext"].apply(lambda x: sia.polarity_scores(x)["compound"])

In [ ]:
df.groupby("macro_topic_clean")["sentiment"].mean().sort_values().plot(kind="barh", figsize=(8,6))